In [1]:
import math
from Utils.hdfs import init
from pyspark.sql import SparkSession, Window, types as T, functions as F
from pyspark.storagelevel import StorageLevel

In [2]:
builder = (
    SparkSession.builder
    .appName("create_dims")
    .master("yarn")
    # .config("spark.yarn.am.memory", "512m")
    # .config("spark.driver.memory", "512m")
    # .config("spark.executor.memory", "512m")
    # .config("spark.executor.cores", "1")
    # .config("spark.executor.instances", "1")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
)
spark = builder.getOrCreate()

In [3]:
init(spark)

In [4]:
%%sql
use warehouse;

""


CREATE TABLE [ IF NOT EXISTS ] table_identifier\
    [ ( col_name1 col_type1 [ COMMENT col_comment1 ], ... ) ]\
    USING data_source\
    [ OPTIONS ( key1=val1, key2=val2, ... ) ]\
    [ PARTITIONED BY ( col_name1, col_name2, ... ) ]\
    [ CLUSTERED BY ( col_name3, col_name4, ... ) \
        [ SORTED BY ( col_name [ ASC | DESC ], ... ) ] \
        INTO num_buckets BUCKETS ]\
    [ LOCATION path ]\
    [ COMMENT table_comment ]\
    [ TBLPROPERTIES ( key1=val1, key2=val2, ... ) ]\
    [ AS select_statement ]

In [5]:
df = spark.read.format('delta').table('warehouse.flight')

In [6]:
df.printSchema()

root
 |-- departure_time: string (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- flight_id: string (nullable = true)
 |-- aircraft_id: string (nullable = true)
 |-- itinerary_no: integer (nullable = true)
 |-- ticket_no: string (nullable = true)
 |-- flight_cost: decimal(10,0) (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- destination_airport: string (nullable = true)
 |-- frequent_flier: boolean (nullable = true)
 |-- travel_date: date (nullable = true)
 |-- airplane_model: string (nullable = true)
 |-- frequent_flier_no: string (nullable = true)
 |-- passenger_name: string (nullable = true)
 |-- passenger_country: string (nullable = true)
 |-- tail_no: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- turbulance: integer (nullable = true)
 |-- temp_at_dept: float (nullable = true)
 |-- fuel_consumed_litre: float (nullable = true)
 |-- taxi_duration_mins: float (nullable = true)
 |-- avg_flight_speed_kmps: double (nullab

In [7]:
dim_airport = df.select(F.col('origin_airport').alias('airport')).distinct()\
                .union(df.select(F.col('destination_airport').alias('airport')).distinct())\
                .distinct().withColumn("id", F.monotonically_increasing_id())\
                .selectExpr(*['id', 'airport'])

In [8]:
dim_airport.write.format('delta').mode('overwrite').saveAsTable('warehouse.dim_airport')

In [ ]:
dim_airport.persist(StorageLevel.MEMORY_AND_DISK_DESER)

In [ ]:
stage_dim_aircraft = df.select('aircraft_id', 'flight_cost', 'origin_airport', 'destination_airport', 
                                 'airplane_model', 'distance', 'fuel_consumed_litre', 'engine_performance')\
                         .join(F.broadcast(dim_airport).alias('a1'), F.col('a1.airport') == F.col('origin_airport'), 'inner')\
                         .join(F.broadcast(dim_airport).alias('a2'), F.col('a2.airport') == F.col('destination_airport'), 'inner')\
                         .select('aircraft_id', 'flight_cost', 'origin_airport', 'destination_airport', 
                                                                    'airplane_model', 'distance', 'fuel_consumed_litre', 'engine_performance', 
                                                                    F.col('a1.id').alias('origin_airport_id'), F.col('a2.id').alias('destination_airport_id'))
dim_aircraft = stage_dim_aircraft.groupBy('aircraft_id', 'airplane_model')\
                                 .agg(F.avg('flight_cost').cast('int').alias('avg_cost'), 
                                      F.sum('distance').alias('total_distance_flown'), 
                                      F.avg('fuel_consumed_litre').alias('avg_fuel_consumed_litre'), 
                                      F.min('engine_performance').alias('min_engine_performance'), 
                                      F.max('engine_performance').alias('max_engine_performance'),
                                      F.percentile_approx('origin_airport_id', 0.7).alias('origin_airport_id'),
                                      F.percentile_approx('destination_airport_id', 0.7).alias('destination_airport_id'))\
                                 .withColumn("id", F.monotonically_increasing_id())

In [ ]:
spark.conf.get("spark.sql.warehouse.dir")

In [ ]:
%%sql
drop table warehouse.dim_aircraft;

In [ ]:
dim_aircraft.coalesce(4).write.format('delta').mode('overwrite').saveAsTable('warehouse.dim_aircraft')

In [ ]:
spark.stop()